# 수정주가를 채우다 — 외부가 안 주는 4년을 어떻게 메웠나

> `notebooks/02-품질·전처리/04.수정주가를채우다.ipynb` · 2026-09-02 · 이동원
> 마이그레이션 v9 · 이슈 [#51](https://github.com/devlee328288/Alpha_Stack/issues/51)

---

## 이 노트북이 답하는 것

> **"삼성전자가 하루 만에 −98% 폭락한 것으로 읽히는데, 어떻게 고쳤나?"**

`daily_price.close` 는 KRX 원문 그대로라 **액면분할이 조정돼 있지 않습니다.**
분할일에 가격이 그대로 뚝 떨어지므로, `close` 로 수익률을 계산하면 폭락으로 읽힙니다.

이 문제를 **다 고쳤습니다.** 다만 고치는 과정에서 계획을 두 번 바꿔야 했고,
그 두 번이 이 노트북의 내용입니다.

| 무엇을 하려 했나 | 무엇이 막았나 | 어떻게 했나 |
|---|---|---|
| FDR 수정주가를 받아 싣는다 | FDR 이 **최근 3,000거래일만** 준다 | 그 앞은 우리 조정계수로 이어 붙였다 |
| FDR 의 시·고·저·종가를 그대로 싣는다 | FDR 이 네 칸을 **따로 반올림**해 `고가 < 종가` 가 된다 | 배율 하나를 넷에 똑같이 곱했다 |

결과부터 적으면 — **9,223,644행 전부(100%)** 채웠고, `fdr` 81.6% · `chain` 18.4% 입니다.

---

## 0. 준비

In [1]:
import sqlite3
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import pandas as pd

from common.paths import krx_db_path

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

conn = sqlite3.connect(f"file:{krx_db_path().as_posix()}?mode=ro", uri=True)
print("DB :", krx_db_path())
print("스키마 v", conn.execute("PRAGMA user_version").fetchone()[0])

DB : C:\Users\kik32\workspace\EST-Camp-AI-Quant\team_project\Alpha_Stack\data\krx_cache.db
스키마 v 9


---

## 1. 문제 — 삼성전자 2018-05-04

액면분할 50:1 이 일어난 날입니다. **원문을 그대로** 뽑아 보겠습니다.

In [2]:
삼성 = pd.read_sql_query(
    """SELECT bas_dd, open, high, low, close, change_rate, volume, listed_shares,
              adj_open, adj_high, adj_low, adj_close, adj_source
       FROM daily_price WHERE code = '005930'
         AND bas_dd BETWEEN '20180426' AND '20180508' ORDER BY bas_dd""",
    conn)
삼성[["bas_dd", "open", "high", "low", "close", "change_rate", "volume", "listed_shares"]]

,bas_dd,open,high,low,close,change_rate,volume,listed_shares
0,20180426,2521000,2608000,2520000,2607000,3.45,360931,128386494
1,20180427,2669000,2682000,2622000,2650000,1.65,606216,128386494
2,20180430,0,0,0,2650000,0.00,0,128386494
3,20180502,0,0,0,2650000,0.00,0,128386494
4,20180503,0,0,0,2650000,0.00,0,128386494
5,20180504,53000,53900,51800,51900,-2.08,39565391,6419324700
6,20180508,52600,53200,51900,52600,1.35,23104720,6419324700


세 가지가 한꺼번에 보입니다.

1. **20180504 에 종가가 2,650,000 → 51,900** 으로 떨어집니다. 그런데 `change_rate` 는 **−2.08%** 입니다.
2. **20180430·0502·0503 은 시·고·저가가 0** 입니다 — 거래정지입니다.
   KRX 는 주권 교체 때문에 분할 전에 반드시 거래를 정지시킵니다.
3. **상장주식수가 128,386,494 → 6,419,324,700** 으로 정확히 50배가 됩니다.

`close` 로 계산한 수익률과 KRX 가 알려 주는 실제 등락률을 나란히 놓아 보겠습니다.

In [3]:
원가격수익률 = 51900 / 2650000 - 1
print(f"close 로 계산     : {원가격수익률 * 100:+.2f}%")
print(f"KRX change_rate  : {삼성.loc[삼성.bas_dd == '20180504', 'change_rate'].iloc[0]:+.2f}%")
print()
print("→ 같은 날에 대한 두 값이 96%p 차이납니다. close 가 미조정 원가격이라 그렇습니다.")

close 로 계산     : -98.04%
KRX change_rate  : -2.08%

→ 같은 날에 대한 두 값이 96%p 차이납니다. close 가 미조정 원가격이라 그렇습니다.


### 규모 — 우리 자료 전체에서

분할·병합은 삼성전자 하나의 일이 아닙니다. **주식수 배율과 가격 배율이 서로 역수인 날**을
분할·병합으로 판정해 전수로 셌습니다.

In [4]:
규모 = pd.read_sql_query(
    """WITH seq AS (
         SELECT code, bas_dd, close, listed_shares,
                LAG(close) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞종가,
                LAG(listed_shares) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞주식수
         FROM daily_price WHERE close > 0 AND listed_shares > 0)
       SELECT COUNT(*) AS 이벤트, COUNT(DISTINCT code) AS 종목
       FROM seq
       WHERE 앞종가 > 0 AND 앞주식수 > 0
         AND ABS((CAST(listed_shares AS REAL) / 앞주식수)
                 * (CAST(close AS REAL) / 앞종가) - 1) < 0.10
         AND (CAST(listed_shares AS REAL) / 앞주식수 > 1.5
              OR CAST(listed_shares AS REAL) / 앞주식수 < 1 / 1.5)""",
    conn)
전체종목 = conn.execute("SELECT COUNT(DISTINCT code) FROM daily_price").fetchone()[0]
print(f"분할·병합 이벤트 {규모.이벤트[0]:,}건 · {규모.종목[0]:,}종 "
      f"(전체 {전체종목:,}종의 {규모.종목[0] / 전체종목 * 100:.1f}%)")

분할·병합 이벤트 665건 · 538종 (전체 3,677종의 14.6%)


---

## 2. 첫 번째 벽 — FDR 이 2010년을 안 준다

FinanceDataReader(MIT · 0.9.202)가 수정주가를 줍니다. 그런데 **범위가 짧습니다.**

`start` 를 2010-01-01 로 줘도 2014-06-13 부터만 옵니다. FDR 의 한국 주식 경로는
네이버 `fchart` 이고 **그 서버가 3,000건에서 자릅니다.** FDR 코드는 이미 `count=6000` 을
보내고 있는데도 그렇습니다 — 직접 확인한 것이 아래입니다.

In [5]:
import re

import requests

for cnt in (3000, 6000, 9000):
    url = ("https://fchart.stock.naver.com/sise.nhn"
           f"?timeframe=day&count={cnt}&requestType=0&symbol=005930")
    items = re.findall(r'<item data="(.*?)" />', requests.get(url, timeout=30).text)
    print(f"count={cnt:>4} → {len(items):,}건 · 첫날 {items[0].split('|')[0]}")

count=3000 → 3,000건 · 첫날 20140613
count=6000 → 3,000건 · 첫날 20140613
count=9000 → 3,000건 · 첫날 20140613


요청을 아무리 키워도 **3,000건**입니다. 우리가 요청을 잘못한 것이 아니라 서버가 자릅니다.

pykrx 의 `get_market_ohlcv(..., adjusted=True)` 도 같은 네이버 경로라 결과가 같습니다.
그래서 **2010-01-04 ~ 2014-06-12 구간에는 외부 수정주가가 없습니다.** 그 크기를 재 보겠습니다.

In [6]:
구멍 = pd.read_sql_query(
    """SELECT
         (SELECT COUNT(*) FROM daily_price) AS 전체,
         (SELECT COUNT(*) FROM daily_price WHERE bas_dd < '20140613') AS 이전,
         (SELECT COUNT(DISTINCT bas_dd) FROM daily_price
            WHERE bas_dd < '20140613') AS 이전거래일""",
    conn)
비율 = 구멍.이전[0] / 구멍.전체[0] * 100
print(f"20140613 이전 : {구멍.이전[0]:,}행 · {구멍.이전거래일[0]:,}거래일 ({비율:.1f}%)")
print()
print("홀드아웃이 20240901 이므로 이 구멍은 **전부 학습구간 안**입니다.")
print("미조정으로 두면 #51 이 지적한 -98% 가 그 구간에 그대로 남습니다.")

20140613 이전 : 2,146,042행 · 1,103거래일 (23.3%)

홀드아웃이 20240901 이므로 이 구멍은 **전부 학습구간 안**입니다.
미조정으로 두면 #51 이 지적한 -98% 가 그 구간에 그대로 남습니다.


### 그래서 — 앵커를 놓고 뒤로 이어 붙였다

FDR 이 닿는 **가장 이른 날**을 앵커로 삼습니다. 그 날의 배율을 FDR 이 알려 주므로,
거기서부터 우리 조정계수를 곱해 과거로 내려갑니다.

```
20100104 ────────────── 20140612 │ 20140613 ────────── 20260901
  chain (계수로 뒤로 이어 붙임)   │      fdr (외부 실측)
                                 ↑
                          앵커. 이 날의 배율을 FDR 이 알려 준다
```

배율의 정의는 `scale[i] = adj_close[i] / close[i]` 이고, 옮기는 규칙은 두 줄입니다.

```
뒤로 (과거 방향)   scale[i] = scale[i+1] × factor[i+1]
앞으로 (미래 방향) scale[i] = scale[i-1] ÷ factor[i]
```

`factor` 는 그 날 일어난 조정이고 **그 앞의 행들에** 적용되므로, 과거로 갈 때 곱하고
미래로 갈 때 나눕니다. 계수 계산 자체는 `common/corporate_actions.py` 에 이미 있었습니다
(평상일은 기준가, 재개일은 상장주식수 배율 — 왜 둘로 나뉘는지는 그 파일 주석에 있습니다).

**배율은 `Fraction` 으로 옮깁니다.** 1/50 같은 계수가 수천 행에 걸쳐 곱해지므로
부동소수로 누적하면 반올림 잡음이 되돌아옵니다.

### 🔴 그런데 이 계산이 맞는지 어떻게 아나

2010~2014 구간은 **대조할 외부 자료가 없습니다.** 그게 애초에 이어 붙인 이유입니다.

그래서 이렇게 쟀습니다 — **FDR 을 일부러 앵커 하루만 남기고 지운 뒤**, 나머지 2,999일을
전부 우리 계산으로 채웁니다. 그리고 진짜 FDR 값과 비교합니다.
그 차이가 곧 **2010~2014 에 우리가 넣은 값의 오차 상한**입니다.

In [7]:
from ingest.clients import fdr_data
from ingest.store import adj_price

ANCHOR = "20140613"
결과 = []
표본 = [("005930", "삼성전자", "20180504", 50),
        ("035420", "NAVER", "20181012", 5),
        ("035720", "카카오", "20210415", 5)]
for code, name, split_dd, ratio in 표본:
    rows = adj_price.load_rows(conn, code)
    adjusted = fdr_data.fetch_adjusted(code)
    # 앵커 하루만 남긴다
    ours = {r[-1]: r[3] for r in adj_price.build_rows(rows, {ANCHOR: adjusted[ANCHOR]})}
    차이 = [abs(ours[d] - v["adj_close"]) / v["adj_close"]
            for d, v in adjusted.items() if v["adj_close"] and ours.get(d)]
    차이.sort()
    결과.append({"종목": f"{name}({code})", "분할일": split_dd,
                 "대조일수": len(차이),
                 "중앙(%)": 차이[len(차이) // 2] * 100,
                 "최대(%)": 차이[-1] * 100, "공지분할": f"{ratio}:1"})
pd.DataFrame(결과)

,종목,대조일수,중앙(%),최대(%),공지분할
0,삼성전자(005930),2999,0.000000,0.000000,50:1
1,NAVER(035420),2999,0.142038,0.142038,5:1
2,카카오(035720),2999,0.037549,0.394557,5:1


**최대 0.39%.** 12년치를 앵커 하나에서 이어 붙였는데 그 정도로 재현됩니다.

NAVER 를 보면 중앙값과 최대값이 같습니다(0.142%). 이건 **상수 배율 차이**라는 뜻이고,
수익률은 비율이라 **상수 배율은 약분돼 사라집니다.** 즉 학습에 쓰는 값에는 영향이 없습니다.

---

## 3. 두 번째 벽 — FDR 이 고가를 종가보다 낮게 준다

처음에는 FDR 이 준 네 칸을 그대로 실었습니다. 그랬더니 표본 3종에서만
**고저 관계 위반 68행**이 나왔습니다. 전부 `fdr` 출처였고, 원가격 위반은 0행이었습니다.

In [8]:
사례 = pd.read_sql_query(
    """SELECT bas_dd, open, high, low, close, adj_open, adj_high, adj_low, adj_close
       FROM daily_price WHERE code = '005930' AND bas_dd = '20150127'""", conn)
print("원문 :", 사례[["open", "high", "low", "close"]].to_dict("records")[0])
print()
print("FDR 이 그대로 주던 값 : adj_high=27,999  adj_close=28,000   ← 고가 < 종가")
print("지금 우리가 싣는 값   :",
      {k: v for k, v in 사례[["adj_open", "adj_high", "adj_low", "adj_close"]]
       .to_dict("records")[0].items()})

원문 : {'open': 1375000, 'high': 1400000, 'low': 1374000, 'close': 1400000}

FDR 이 그대로 주던 값 : adj_high=27,999  adj_close=28,000   ← 고가 < 종가
지금 우리가 싣는 값   : {'adj_open': 27500.0, 'adj_high': 28000.0, 'adj_low': 27480.0, 'adj_close': 28000.0}


원문에서는 `high == close == 1,400,000` 으로 **같습니다.**
FDR 은 네 칸을 각각 따로 반올림하기 때문에 `1,400,000 / 50 = 28,000` 이 되어야 할 고가가
27,999 로 내려앉았습니다.

`true_range`·`parkinson_20` 처럼 고저 폭을 쓰는 피처는 여기서 **음수**를 뱉습니다.

**고친 방법은 단순합니다** — 배율 하나를 네 칸에 똑같이 곱합니다.
하루 안의 시·고·저·종가는 같은 스케일이므로, 이렇게 하면 그 날의 고저 폭과 시종 관계가
**정확히 보존됩니다.** 뒤집히던 칸만 제자리를 찾고 나머지는 FDR 값과 그대로 일치합니다.

In [9]:
위반 = conn.execute(
    """SELECT COUNT(*) FROM daily_price
       WHERE adj_high IS NOT NULL AND adj_low IS NOT NULL AND adj_close IS NOT NULL
         AND (adj_close > adj_high + 0.01 OR adj_close < adj_low - 0.01)""").fetchone()[0]
print(f"전체 9,223,644행에서 고저 관계 위반: {위반:,}행")

전체 9,223,644행에서 고저 관계 위반: 0행


---

## 4. 정지일 — 0 을 가격으로 실으면 안 된다

거래정지일은 `open = high = low = 0` 이고 종가만 직전 값을 물고 있습니다.
FDR 도 정확히 같은 모양으로 줍니다.

0 에 배율을 곱하면 0 이 되어 **"그 날 가격이 0원이었다"** 가 됩니다. 수익률은 −100% 가 되고,
고저 검사도 `0 ≤ 0 ≤ 0` 이라 통과해 버립니다. 그래서 **시·고·저가는 비우고 종가만 채웁니다.**

In [10]:
삼성[["bas_dd", "open", "high", "low", "close",
      "adj_open", "adj_high", "adj_low", "adj_close", "adj_source"]]

,bas_dd,open,high,low,close,adj_open,adj_high,adj_low,adj_close,adj_source
0,20180426,2521000,2608000,2520000,2607000,50420.0,52160.0,50400.0,52140.0,fdr
1,20180427,2669000,2682000,2622000,2650000,53380.0,53640.0,52440.0,53000.0,fdr
2,20180430,0,0,0,2650000,NaN,NaN,NaN,53000.0,fdr
3,20180502,0,0,0,2650000,NaN,NaN,NaN,53000.0,fdr
4,20180503,0,0,0,2650000,NaN,NaN,NaN,53000.0,fdr
5,20180504,53000,53900,51800,51900,53000.0,53900.0,51800.0,51900.0,fdr
6,20180508,52600,53200,51900,52600,52600.0,53200.0,51900.0,52600.0,fdr


20180430·0502·0503 의 `adj_open`·`adj_high`·`adj_low` 가 `None` 이고 `adj_close` 는 살아 있습니다.
그 종가가 **재개일 등락률의 기준**이 되기 때문에 버리면 안 됩니다.

이제 분할일 수익률이 어떻게 됐는지 보겠습니다.

In [11]:
전 = 삼성.loc[삼성.bas_dd == "20180503"].iloc[0]
후 = 삼성.loc[삼성.bas_dd == "20180504"].iloc[0]
print(f"close     로 : {후.close / 전.close - 1:+.4f}  ({(후.close / 전.close - 1) * 100:+.2f}%)")
print(f"adj_close 로 : {후.adj_close / 전.adj_close - 1:+.4f}  "
      f"({(후.adj_close / 전.adj_close - 1) * 100:+.2f}%)")
print(f"KRX 실제      : {후.change_rate:+.2f}%")

close     로 : -0.9804  (-98.04%)
adj_close 로 : -0.0208  (-2.08%)
KRX 실제      : -2.08%


---

## 5. 뜻밖의 소득 — 3,000일 창은 종목마다 따로 걸린다

적재를 끝내고 출처를 세어 보니 **`fdr` 이 2010년에도 있었습니다.**
FDR 이 3,000일만 준다면 있을 수 없는 일입니다.

이유는 이렇습니다 — **3,000일 창은 오늘이 아니라 "그 종목의 마지막 거래일"에 걸립니다.**
2015년에 상장폐지된 종목은 그 시점에서 3,000일을 거슬러 주므로 2010년이 들어옵니다.

In [12]:
출처 = pd.read_sql_query(
    """SELECT adj_source AS 출처, COUNT(*) AS 행,
              MIN(bas_dd) AS 첫날, MAX(bas_dd) AS 마지막날
       FROM daily_price WHERE adj_source IS NOT NULL
       GROUP BY adj_source ORDER BY 행 DESC""", conn)
출처["비율(%)"] = (출처.행 / 출처.행.sum() * 100).round(1)
출처

,출처,행,첫날,마지막날,비율(%)
0,fdr,7522431,20100104,20260901,81.6
1,chain,1701213,20100104,20140612,18.4


In [13]:
이전 = pd.read_sql_query(
    """SELECT adj_source AS 출처, COUNT(*) AS 행, COUNT(DISTINCT code) AS 종목
       FROM daily_price WHERE bas_dd < '20140613' GROUP BY adj_source""", conn)
print("20140613 이전 구간의 출처 —")
display(이전)
print("소멸 종목은 자기 마지막 거래일 기준으로 3,000일을 받으므로 2010년이 들어옵니다.")

20140613 이전 구간의 출처 —


,출처,행,종목
0,chain,1701213,1679
1,fdr,444829,686


소멸 종목은 자기 마지막 거래일 기준으로 3,000일을 받으므로 2010년이 들어옵니다.


---

## 6. 검증 — 네 가지로 본다

**행 수만 세면 안 됩니다.** 칸을 덮어써서 값이 사라져도 행 수는 그대로입니다.
(2026-09-02 재무에서 PK 에 칸 하나를 빠뜨려 6.4%가 조용히 사라진 적이 있습니다.)

In [14]:
검증 = pd.read_sql_query(
    """WITH seq AS (
         SELECT code, bas_dd, adj_close, change_rate,
                LAG(adj_close) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞수정
         FROM daily_price WHERE adj_close IS NOT NULL AND close > 0)
       SELECT COUNT(*) AS 쌍,
              SUM(CASE WHEN ABS((adj_close / 앞수정 - 1) * 100 - change_rate) > 0.15
                       THEN 1 ELSE 0 END) AS 어긋남
       FROM seq WHERE 앞수정 > 0 AND change_rate IS NOT NULL""", conn)
비율 = 검증.어긋남[0] / 검증.쌍[0] * 100
print(f"① 분할일 갭  {검증.쌍[0]:,}쌍 중 {검증.어긋남[0]:,}쌍 어긋남 ({비율:.3f}%)")
print(f"② 고저 관계  위반 {위반:,}행")
print("③ 0·음수    ",
      conn.execute("""SELECT COUNT(*) FROM daily_price WHERE adj_open <= 0
                      OR adj_high <= 0 OR adj_low <= 0 OR adj_close <= 0""").fetchone()[0], "행")
채움 = conn.execute(
    'SELECT COUNT(*) FROM daily_price WHERE adj_close IS NOT NULL').fetchone()[0]
전체 = conn.execute('SELECT COUNT(*) FROM daily_price').fetchone()[0]
print("④ 채움률    ",
      f"{채움:,} / {전체:,}")


① 분할일 갭  9,219,967쌍 중 13,965쌍 어긋남 (0.151%)
② 고저 관계  위반 0행


③ 0·음수     0 행


④ 채움률     9,223,644 / 9,223,644


### 어긋난 0.151% 는 무엇인가

**그냥 넘기지 않고 성격을 특정했습니다.** 어긋난 행의 가격 수준을 전체와 비교해 봅니다.

In [15]:
분포 = pd.read_sql_query(
    """WITH seq AS (
         SELECT code, bas_dd, adj_close, change_rate, close,
                LAG(adj_close) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞수정,
                LAG(close) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞종가,
                LAG(open) OVER (PARTITION BY code ORDER BY bas_dd) AS 앞시가
         FROM daily_price WHERE adj_close IS NOT NULL AND close > 0)
       SELECT 앞종가, 앞시가, ABS((adj_close / 앞수정 - 1) * 100 - change_rate) AS 차이
       FROM seq WHERE 앞수정 > 0 AND change_rate IS NOT NULL
         AND ABS((adj_close / 앞수정 - 1) * 100 - change_rate) > 0.15""", conn)
전체중앙 = conn.execute(
    """SELECT close FROM daily_price WHERE close > 0 ORDER BY close
       LIMIT 1 OFFSET (SELECT COUNT(*) / 2 FROM daily_price WHERE close > 0)""").fetchone()[0]

print(f"어긋난 {len(분포):,}행 —")
print(f"  차이 중앙값        {분포.차이.median():.3f}%p · 75분위 {분포.차이.quantile(.75):.3f}%p")
print(f"  1%p 초과           {(분포.차이 > 1).sum():,}행 "
      f"(전체의 {(분포.차이 > 1).sum() / 검증.쌍[0] * 100:.4f}%)")
print(f"  🔴 전일 종가 중앙값 {분포.앞종가.median():,.0f}원  ← 전체 중앙값 {전체중앙:,}원")
print()
print("저가주에서 1원 반올림이 커 보이는 것입니다 —")
print(f"1,060원에서 1원은 {1 / 1060 * 100:.3f}% 라 이틀이면 0.15%p 문턱을 넘습니다.")

어긋난 13,965행 —
  차이 중앙값        0.270%p · 75분위 0.499%p
  1%p 초과           1,430행 (전체의 0.0155%)
  🔴 전일 종가 중앙값 1,070원  ← 전체 중앙값 6,400원

저가주에서 1원 반올림이 커 보이는 것입니다 —
1,060원에서 1원은 0.094% 라 이틀이면 0.15%p 문턱을 넘습니다.


---

## 7. 실측 거래일 달력

같은 마이그레이션에서 `trading_calendar` 표를 만들었습니다.

**휴장일을 계산으로 맞히지 않습니다.** 주말만 걸러 세면 개발구간 평일 3,042일 중
162일(5.3%)이 어긋나고, 그 162일은 명절·공휴일이라 하필 실적 발표와 뉴스가 몰립니다.
우리가 **실제로 받은 날**이 거래일입니다 — 추정이 아니라 기록입니다.

로직 자체는 `common/trading_calendar.py` 에 이미 있었습니다. 표로 옮긴 이유는 속도입니다.

In [16]:
import time

import common.trading_calendar as tc

t = time.perf_counter()
days = tc.load_session_days(krx_db_path(), refresh=True)
표 = time.perf_counter() - t

t = time.perf_counter()
원본 = conn.execute("SELECT DISTINCT bas_dd FROM daily_price").fetchall()
훑기 = time.perf_counter() - t

print(f"trading_calendar 표 : {표 * 1000:6.1f}ms · {len(days):,}일")
print(f"daily_price 훑기     : {훑기 * 1000:6.1f}ms · {len(원본):,}일")
print(f"두 답이 같은가       : {frozenset(str(r[0]) for r in 원본) == days}")
print(f"빨라진 배수          : {훑기 / 표:.0f}배")

trading_calendar 표 :   17.8ms · 4,102일
daily_price 훑기     :  643.2ms · 4,102일
두 답이 같은가       : True
빨라진 배수          : 36배


In [17]:
달력 = pd.read_sql_query(
    """SELECT market AS 시장, COUNT(*) AS 거래일,
              MIN(bas_dd) AS 첫날, MAX(bas_dd) AS 마지막날
       FROM trading_calendar GROUP BY market""", conn)
달력

,시장,거래일,첫날,마지막날
0,ALL,4102,20100104,20260901
1,KOSDAQ,4102,20100104,20260901
2,KOSPI,4102,20100104,20260901


---

## 8. 팀에 전하는 것

| 하던 것 | 앞으로 |
|---|---|
| `close` 로 수익률 계산 | **`adj_close`** 로 계산합니다 |
| `high`·`low` 로 변동성 피처 | **`adj_high`·`adj_low`** 를 씁니다 |
| 시가총액 | **`close`** 그대로입니다 (`market_cap = close × listed_shares` 는 원가격이라야 맞습니다) |

`adj_source` 로 그 행의 값이 어디서 왔는지 확인할 수 있습니다 — `fdr` 이면 외부 실측,
`chain` 이면 우리가 계수로 이어 붙인 값입니다.

### ⚠️ 알고 써야 하는 성질 — 후방조정은 과거가 바뀐다

수정주가는 **현재 가격 기준으로 과거를 눌러 놓은 값**입니다. 그래서 **새 액면분할이
하나 생기면 그 종목의 과거 값이 전부 바뀝니다.** 이건 우리 구현의 문제가 아니라
후방조정 자체의 성질이고, FDR 도 같습니다.

그래서 두 가지를 했습니다.

- **원 가격 칸을 덮지 않았습니다.** 원문은 그대로 있으므로 언제든 되돌아갈 수 있습니다.
- 종목마다 **언제 계산했는지**를 `collect_log` 에 남겼습니다 (`source='adj_price'`).

학습·라벨처럼 재현이 중요한 자리에서는 **구간만 보는** `corporate_actions.span_factor` 를
쓰는 편이 안전합니다. 그쪽은 구간 밖의 조정을 보지 않으므로 뒤에 새 분할이 생겨도
이미 계산한 값이 변하지 않습니다.

In [18]:
대장 = pd.read_sql_query(
    """SELECT COUNT(*) AS 종목, MIN(last_success_at) AS 처음, MAX(last_success_at) AS 마지막
       FROM collect_log WHERE source = 'adj_price'""", conn)
print("수정주가 계산 이력 (collect_log) —")
display(대장)
conn.close()

수정주가 계산 이력 (collect_log) —


,종목,처음,마지막
0,3677,2026-09-02T16:34:51+09:00,2026-09-02T16:56:49+09:00
